In [2]:
import pandas as pd
import numpy as np
import linopy

In [6]:
marginal_costs = pd.Series([0, 30, 60, 80], index=["Wind", "Coal", "Gas", "Oil"])
marginal_costs

capacities = pd.Series([3000, 35000, 8000, 2000], index=["Wind", "Coal", "Gas", "Oil"])
capacities

load = 42000

m = linopy.Model()

g = m.add_variables(lower=0, upper=capacities, coords=[capacities.index], name="g")
g

m.add_objective(marginal_costs.values * g, sense="min")
m.objective

m.add_constraints(g.sum() == load, name="energy_balance")

m.solve(solver = "highs")

m.solution.to_dataframe()

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-wddp4lw6 has 1 row; 4 cols; 4 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [3e+01, 8e+01]
  Bound   [2e+03, 4e+04]
  RHS     [4e+04, 4e+04]
Presolving model
1 rows, 3 cols, 3 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-4); nonzeros 0(-4) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-wddp4lw6
Model status        : Optimal
Objective value     :  1.2900000000e+06
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


<positron-console-cell-6>:14: LinopySemanticsWarning: Coordinate mismatch in this operator's constant operand silently aligned by legacy (by label, the coefficient being 0 at the variable's labels it lacks). Under v1 this raises ValueError.
  Dim:       'dim_0': left=['Wind', 'Coal', 'Gas', 'Oil'], right=[0, 1, 2, 3]
  Resolve:   `.sel(...)` / `.reindex(...)` to align
             `.assign_coords(...)` to relabel one side
             `linopy.align(...)` to pre-align several operands
             or pass an explicit `join=` argument.
  Opt in:    linopy.options['semantics'] = 'v1'
  Silence:   warnings.filterwarnings('ignore', category=LinopySemanticsWarning)


,g
dim_0,
Wind,3000.0
Coal,35000.0
Gas,4000.0
Oil,0.0


In [25]:
generators = ["Coal", "Wind", "Gas", "Oil", "Hydro"]
countries = ["South_Africa", "Mozambique"]

capacities = pd.DataFrame(
    {
        "Coal": [35000, 0],
        "Wind": [3000, 0],
        "Gas": [8000, 0],
        "Oil": [2000, 0],
        "Hydro": [0, 1200],
    },
    index=countries,
)
capacities.index.name = "countries"
capacities.columns.name = "generators"

capacities

marginal_costs = pd.Series([30, 0, 60, 80, 0], index=generators)
marginal_costs.index.name = "generators"
marginal_costs

load = pd.Series([42000, 650], index=countries)
load.index.name = "countries"
load


m = linopy.Model()

g = m.add_variables(lower=0, upper=capacities, name="g")
g

transmission = 500
f = m.add_variables(lower=-transmission, upper=transmission, name="flow_MZ_SA")
f

Variable
--------
flow_MZ_SA ∈ [-500, 500]

In [26]:
for country in countries:
    sign = -1 if country == "Mozambique" else 1  # minimal incidence matrix
    m.add_constraints(
        g.loc[country].sum() + sign * f == load[country],
        name=f"{country}_KCL",
    )

m.constraints["Mozambique_KCL"]

Constraint `Mozambique_KCL`
---------------------------
+1 g[Mozambique, Coal] + 1 g[Mozambique, Wind] + 1 g[Mozambique, Gas] + 1 g[Mozambique, Oil] + 1 g[Mozambique, Hydro] - 1 flow_MZ_SA = 650.0

In [39]:

obj = (g * marginal_costs).sum()

m.add_objective(obj, sense="min", overwrite=True)

m.solve(solver = "highs")

g.solution.to_dataframe()

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-rt0wkcae has 2 rows; 11 cols; 12 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [3e+01, 8e+01]
  Bound   [5e+02, 4e+04]
  RHS     [6e+02, 4e+04]
Presolving model
1 rows, 4 cols, 4 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-2); columns 0(-11); nonzeros 0(-12) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-rt0wkcae
Model status        : Optimal
Objective value     :  1.2600000000e+06
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


solution
countries    generators          
South_Africa Coal         35000.0
             Wind          3000.0
             Gas           3500.0
             Oil              0.0
             Hydro            0.0
Mozambique   Coal             0.0
             Wind             0.0
             Gas              0.0
             Oil              0.0
             Hydro         1150.0

In [56]:
A = np.array([[2,-1,0],[1,-3,1],[0,1,-2]])
b = np.array([0,0,-1])
thetas = np.linalg.solve(A,b)

col1 = np.array([3,1,1,5,2])/8
col2 = np.array([-1,1,1,1,2])/4
col3 = np.array([ thetas[0] - thetas[1], thetas[1] - thetas[2], thetas[2], thetas[0], thetas[1]])

ptdf = np.column_stack( (col1, col2, col3) )

ptdf[:,0] - ptdf[:,2]

ptdf

array([[ 0.375, -0.25 , -0.125],
       [ 0.125,  0.25 , -0.375],
       [ 0.125,  0.25 ,  0.625],
       [ 0.625,  0.25 ,  0.125],
       [ 0.25 ,  0.5  ,  0.25 ]])

In [51]:
Ap = np.array([[2,-1,0],[1,-2,1],[0,1,-2]])
b = np.array([1,0,0])
thetas = np.linalg.solve(Ap,b)

thetas

np.array([ thetas[0] - thetas[1], thetas[1] - thetas[2], thetas[2], thetas[0], thetas[1]])

array([0.25, 0.25, 0.25, 0.75, 0.5 ])

In [87]:
# Define the coefficient matrix (A) and the ordinate values (b)
A = np.array([[1.0, 1.0], 
              [3/4, 1/4]])
b = np.array([17, 10.5])

# Solve for [gA, gB]
optimal_gA, optimal_gB = np.linalg.solve(A, b)
min_cost = (optimal_gA * 25) + (optimal_gB * 85)



print(optimal_gA)
print(optimal_gB)
print(min_cost)

12.5
4.5
695.0


In [73]:
16* 3/4



12.0